# Gradient Boosted Survival Model
**In this notebook, we'll train and test a gradient boosting survival model from scikit-learn to predict 6 month survival from time of first line treatment for advanced non-small cell lung cancer.**

In [1]:
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import make_scorer

from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import cumulative_dynamic_auc

import matplotlib.pyplot as plt

from utils.cv_predictions_lung import cross_validated_survival_predictions, out_of_fold_auc

## Import

In [2]:
dtype_map = pd.read_csv('../outputs/1L_features_training_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
train_df = pd.read_csv('../outputs/1L_features_training.csv', dtype = dtype_map)
train_df.shape

(66187, 161)

In [3]:
dtype_map = pd.read_csv('../outputs/1L_features_testing_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
test_df = pd.read_csv('../outputs/1L_features_testing.csv', dtype = dtype_map)
test_df.shape

(16552, 161)

In [4]:
censored_before_6m = (len(train_df.query('event == 0 and duration < 180', engine = 'python'))/len(train_df))*100
censored_before_12m = (len(train_df.query('event == 0 and duration < 365', engine = 'python'))/len(train_df))*100

print(f'Percentage censored before 6 months: {round(censored_before_6m, 1)}%')
print(f'Percentage censored before 12 months: {round(censored_before_12m, 1)}%')

Percentage censored before 6 months: 6.7%
Percentage censored before 12 months: 10.3%


## Preprocess

### Selecting features

#### Categorical variables

In [5]:
list(train_df.select_dtypes(include = ['category']).columns)

['Histology',
 'SmokingStatus',
 'adv_diagnosis_year',
 'Ethnicity_mod',
 'Race_mod',
 'region',
 'EGFR_status',
 'KRAS_status',
 'BRAF_status',
 'ALK_status',
 'ROS1_status',
 'MET_status',
 'RET_status',
 'NTRK_status',
 'PDL1_status',
 'PDL1_percent_staining']

In [6]:
# Percent unknown
var_dict = {}
for var in train_df.select_dtypes(include=['category']).columns:
    value_props = train_df[var].value_counts(normalize=True)
    # Combine 'unknown' and 'Unknown' if both exist
    unknown_pct = value_props.get('unknown', 0) + value_props.get('Unknown', 0) + value_props.get('Unknown/Not documented', 0)
    var_dict[var] = round(unknown_pct*100, 3)

var_dict

{'Histology': 0,
 'SmokingStatus': np.float64(0.53),
 'adv_diagnosis_year': 0,
 'Ethnicity_mod': 0,
 'Race_mod': 0,
 'region': np.float64(25.685),
 'EGFR_status': np.float64(33.327),
 'KRAS_status': np.float64(52.267),
 'BRAF_status': np.float64(53.231),
 'ALK_status': np.float64(37.119),
 'ROS1_status': np.float64(49.838),
 'MET_status': np.float64(60.201),
 'RET_status': np.float64(59.368),
 'NTRK_status': np.float64(65.623),
 'PDL1_status': np.float64(52.255),
 'PDL1_percent_staining': np.float64(58.977)}

In [7]:
# Percent positive
var_dict = {}
for var in train_df.select_dtypes(include=['category']).columns.drop(['Histology', 'SmokingStatus', 'adv_diagnosis_year', 'Ethnicity_mod', 'Race_mod', 'region', 'PDL1_percent_staining']):
    value_props = train_df[var].value_counts(normalize=True)
    pos_pct = value_props.get('positive', 0)
    var_dict[var] = round(pos_pct*100, 3)

var_dict

{'EGFR_status': np.float64(11.196),
 'KRAS_status': np.float64(13.518),
 'BRAF_status': np.float64(2.697),
 'ALK_status': np.float64(2.08),
 'ROS1_status': np.float64(0.494),
 'MET_status': np.float64(2.652),
 'RET_status': np.float64(0.663),
 'NTRK_status': np.float64(0.499),
 'PDL1_status': np.float64(32.269)}

In [8]:
categorical_var = [
    'Histology',
    'SmokingStatus',
    'EGFR_status',
    'KRAS_status',
    'BRAF_status',
    'ALK_status',
    'MET_status',
    'PDL1_status',
    'PDL1_percent_staining',
]

#### Binary variables 

In [9]:
list(train_df.select_dtypes(include = ['int']).columns)

['days_adv_to_treatment',
 'GroupStage_mod_na',
 'age',
 'sex_male',
 'ecog_newly_gte2',
 'ecog_index_na',
 'hypotension',
 'tachycardia',
 'fevers',
 'hypoxemia',
 'anticoagulant',
 'opioid',
 'steroid',
 'antibiotic',
 'diabetic_med',
 'antidepressant',
 'bone_therapy_agent',
 'immunosuppressant',
 'chf',
 'cardiac_arrhythmias',
 'valvular_disease',
 'pulm_circulation',
 'pvd',
 'htn_uncomplicated',
 'htn_complicated',
 'paralysis',
 'other_neuro',
 'chronic_pulm_disease',
 'diabetes_uncomplicated',
 'diabetes_complicated',
 'hypothyroid',
 'renal_failure',
 'liver_disease',
 'pud',
 'aids_hiv',
 'lymphoma',
 'rheumatic',
 'coagulopathy',
 'obesity',
 'weight_loss',
 'fluid',
 'blood_loss_anemia',
 'deficiency_anemia',
 'alcohol_abuse',
 'drug_abuse',
 'psychoses',
 'depression',
 'van_walraven_score',
 'lymph_met',
 'thoracic_met',
 'liver_met',
 'bone_met',
 'brain_met',
 'adrenal_met',
 'other_viscera_met',
 'other_met',
 'event']

In [10]:
# Did not include days_adv_to_treatment, age, van_walraven, or event
bin_list = [
     'GroupStage_mod_na',
     'sex_male',
     'ecog_newly_gte2',
     'ecog_index_na',
     'hypotension',
     'tachycardia',
     'fevers',
     'hypoxemia',
     'anticoagulant',
     'opioid',
     'steroid',
     'antibiotic',
     'diabetic_med',
     'antidepressant',
     'bone_therapy_agent',
     'immunosuppressant',
     'chf',
     'cardiac_arrhythmias',
     'valvular_disease',
     'pulm_circulation',
     'pvd',
     'htn_uncomplicated',
     'htn_complicated',
     'paralysis',
     'other_neuro',
     'chronic_pulm_disease',
     'diabetes_uncomplicated',
     'diabetes_complicated',
     'hypothyroid',
     'renal_failure',
     'liver_disease',
     'pud',
     'aids_hiv',
     'lymphoma',
     'rheumatic',
     'coagulopathy',
     'obesity',
     'weight_loss',
     'fluid',
     'blood_loss_anemia',
     'deficiency_anemia',
     'alcohol_abuse',
     'drug_abuse',
     'psychoses',
     'depression',
     'lymph_met',
     'thoracic_met',
     'liver_met',
     'bone_met',
     'brain_met',
     'adrenal_met',
     'other_viscera_met',
     'other_met'
]

In [11]:
# Identify binary variables with >3% prevalence 
bin_var_list = []
for bin_var in bin_list:
    if (train_df[bin_var].value_counts(normalize = True)[1] >= 0.03):
        bin_var_list.append(bin_var)

In [12]:
bin_var_list

['sex_male',
 'ecog_newly_gte2',
 'ecog_index_na',
 'hypotension',
 'tachycardia',
 'hypoxemia',
 'opioid',
 'steroid',
 'antibiotic',
 'bone_therapy_agent',
 'chf',
 'cardiac_arrhythmias',
 'pulm_circulation',
 'pvd',
 'htn_uncomplicated',
 'chronic_pulm_disease',
 'diabetes_uncomplicated',
 'diabetes_complicated',
 'hypothyroid',
 'renal_failure',
 'liver_disease',
 'coagulopathy',
 'obesity',
 'weight_loss',
 'fluid',
 'deficiency_anemia',
 'depression',
 'lymph_met',
 'thoracic_met',
 'liver_met',
 'bone_met',
 'brain_met',
 'other_met']

In [13]:
# ecog_index_na added as flags later
binary_var = [
     'sex_male',
     'ecog_newly_gte2',
     'hypotension',
     'tachycardia',
     'hypoxemia',
     'opioid',
     'steroid',
     'antibiotic',
     'bone_therapy_agent',
     'chf',
     'cardiac_arrhythmias',
     'pulm_circulation',
     'pvd',
     'htn_uncomplicated',
     'chronic_pulm_disease',
     'diabetes_uncomplicated',
     'diabetes_complicated',
     'hypothyroid',
     'renal_failure',
     'liver_disease',
     'coagulopathy',
     'obesity',
     'weight_loss',
     'fluid',
     'deficiency_anemia',
     'depression',
     'lymph_met',
     'thoracic_met',
     'liver_met',
     'bone_met',
     'brain_met',
     'other_met'
]

#### Numerical variables

In [14]:
list(train_df.select_dtypes(include = ['float64']).columns)

['GroupStage_mod',
 'days_diagnosis_to_adv',
 'ecog_index',
 'weight_index',
 'bmi_index',
 'percent_change_weight',
 'albumin',
 'alp',
 'alt',
 'anc',
 'ast',
 'bicarbonate',
 'bun',
 'calcium',
 'chloride',
 'creatinine',
 'hemoglobin',
 'platelet',
 'potassium',
 'sodium',
 'total_bilirubin',
 'wbc',
 'albumin_max',
 'alp_max',
 'alt_max',
 'anc_max',
 'ast_max',
 'bicarbonate_max',
 'bun_max',
 'calcium_max',
 'chloride_max',
 'creatinine_max',
 'hemoglobin_max',
 'platelet_max',
 'potassium_max',
 'sodium_max',
 'total_bilirubin_max',
 'wbc_max',
 'albumin_min',
 'alp_min',
 'alt_min',
 'anc_min',
 'ast_min',
 'bicarbonate_min',
 'bun_min',
 'calcium_min',
 'chloride_min',
 'creatinine_min',
 'hemoglobin_min',
 'platelet_min',
 'potassium_min',
 'sodium_min',
 'total_bilirubin_min',
 'wbc_min',
 'albumin_std',
 'alp_std',
 'alt_std',
 'anc_std',
 'ast_std',
 'bicarbonate_std',
 'bun_std',
 'calcium_std',
 'chloride_std',
 'creatinine_std',
 'hemoglobin_std',
 'platelet_std',
 'po

In [15]:
# Added age, van_walraven_score, and days_adv_to_treatment here 
numerical_var = [
    'GroupStage_mod',
    'days_diagnosis_to_adv',
    'ecog_index',
    'weight_index',
    'bmi_index',
    'percent_change_weight',
    'albumin',
    'alp',
    'alt',
    'anc',
    'ast',
    'bicarbonate',
    'bun',
    'calcium',
    'chloride',
    'creatinine',
    'hemoglobin',
    'platelet',
    'potassium',
    'sodium',
    'total_bilirubin',
    'wbc',
    'alp_max',
    'alt_max',
    'ast_max',
    'calcium_max',
    'creatinine_max',
    'platelet_max',
    'potassium_max',
    'sodium_max',
    'total_bilirubin_max',
    'wbc_max',
    'albumin_min',
    'anc_min',
    'bicarbonate_min',
    'calcium_min',
    'hemoglobin_min',
    'platelet_min',
    'potassium_min',
    'sodium_min',
    'wbc_min',
    'age', 
    'van_walraven_score', 
    'days_adv_to_treatment'
]

In [16]:
all_var = categorical_var + binary_var + numerical_var

In [17]:
missing = train_df[all_var].isnull().sum()
missing[missing > 0].sort_values(ascending = False)

van_walraven_score       28252
anc                      16065
anc_min                  15667
chloride                 14592
albumin                  12363
albumin_min              11803
total_bilirubin          11586
alt                      11313
ast                      11113
alp                      11073
bicarbonate              11066
calcium                  11062
total_bilirubin_max      11006
alt_max                  10732
potassium                10631
sodium                   10577
ast_max                  10529
bicarbonate_min          10521
calcium_min              10504
calcium_max              10504
alp_max                  10499
bun                      10265
potassium_max            10093
potassium_min            10093
sodium_max               10035
sodium_min               10035
creatinine                8158
percent_change_weight     7833
creatinine_max            7680
wbc                       6914
platelet                  6831
hemoglobin                6805
wbc_min 

In [18]:
# Create a missing flag
for x in missing[missing > 0].index:
    train_df[f'{x}_na'] = np.where(train_df[x].isna(), 1, 0)
    test_df[f'{x}_na'] = np.where(test_df[x].isna(), 1, 0)

In [19]:
# Add missing flags to binary_var (this includes ecog_index_na and GroupStage_mod_na)
binary_var = binary_var + [col for col in train_df.columns if col.endswith('_na')]

In [20]:
all_var = categorical_var + binary_var + numerical_var

In [21]:
len(all_var)

126

### Setting up Y

In [22]:
y_dtype = [('event', bool), ('duration', float)]

In [23]:
# Death status and time until event needs to be stored as a structured array to be compatible with scikit-survival
train_y = np.array([tuple(x) for x in train_df[['event', 'duration']].values], 
                   dtype = y_dtype)

test_y = np.array([tuple(x) for x in test_df[['event', 'duration']].values], 
                  dtype = y_dtype)

### Creating mini-training set for hyperparameter tuning
**Given the computational cost of hyperparameter search on the full training set, a stratified subsample of 20,000 patients is used to identify optimal hyperparameters. The final model is then trained on the complete training set using these hyperparameters before evaluating performance on the held-out test set.**

In [24]:
train_subsample_df, _ = train_test_split(
    train_df,
    train_size = 20000,
    stratify = train_df['event'],  
    random_state = 42
)

In [25]:
train_subsample_y = np.array([tuple(x) for x in train_subsample_df[['event', 'duration']].values],
                             dtype = y_dtype)

### Creating pipeline

In [26]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median'))
])

categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown = "ignore", sparse_output = False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_var),
        ("cat", categorical_pipeline, categorical_var),
        ("pass", "passthrough", binary_var),
    ],
    remainder="drop"
)

## Train model

In [27]:
def six_month_auc_scorer_approximate(estimator, X, y):
    """
    Approximate 180d AUC using test set for censoring estimation.
    This is not ideal but may be acceptable if censoring is light (eg., <20%).
    """
    chf_funcs = estimator.predict_cumulative_hazard_function(X)
    risk_scores = np.array([chf(180) for chf in chf_funcs])
    
    # Use same y for both (not ideal, but functional)
    auc, _ = cumulative_dynamic_auc(y, y, risk_scores, 180)
    return auc[0]

auc_scorer = make_scorer(
    six_month_auc_scorer_approximate,
    greater_is_better = True,
    needs_proba = False  
)

param_distributions = {
    'model__n_estimators': [400, 700],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [2, 3, 4],
    'model__subsample': [0.75, 0.85],
    'model__max_features': [0.7, 0.9]
}

gb = GradientBoostingSurvivalAnalysis(random_state = 42)

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', gb)
])

search = RandomizedSearchCV(
    estimator = full_pipeline, 
    param_distributions = param_distributions, 
    n_iter = 15, 
    cv = 3,
    scoring = six_month_auc_scorer_approximate,
    refit = False,
    n_jobs = -1,
    random_state = 42,
    verbose = 1
)

search.fit(train_subsample_df[all_var], train_subsample_y)

print(f"Best params: {search.best_params_}")

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Best params: {'model__subsample': 0.85, 'model__n_estimators': 700, 'model__max_features': 0.7, 'model__max_depth': 4, 'model__learning_rate': 0.05}


In [28]:
results_df = pd.DataFrame(search.cv_results_)
results_df.sort_values('rank_test_score', ascending = True).head(10)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__subsample,param_model__n_estimators,param_model__max_features,param_model__max_depth,param_model__learning_rate,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
7,1790.030263,5.591010,0.292351,0.019436,0.85,700,0.7,4,0.05,"{'model__subsample': 0.85, 'model__n_estimator...",0.757673,0.764127,0.770028,0.763942,0.005046,1
6,970.183811,2.918728,0.271050,0.044959,0.75,400,0.9,3,0.05,"{'model__subsample': 0.75, 'model__n_estimator...",0.758691,0.761231,0.769394,0.763105,0.004566,2
12,1684.307355,3.746914,0.235755,0.009687,0.75,700,0.9,2,0.05,"{'model__subsample': 0.75, 'model__n_estimator...",0.757939,0.761698,0.768973,0.762870,0.004580,3
0,1740.105666,3.763959,0.254168,0.007705,0.85,700,0.7,2,0.10,"{'model__subsample': 0.85, 'model__n_estimator...",0.756005,0.760839,0.769891,0.762245,0.005756,4
10,965.353259,1.383302,0.238476,0.004651,0.75,400,0.7,3,0.05,"{'model__subsample': 0.75, 'model__n_estimator...",0.757045,0.760996,0.767561,0.761868,0.004337,5
11,1769.837960,3.257952,0.247081,0.001387,0.85,700,0.7,2,0.05,"{'model__subsample': 0.85, 'model__n_estimator...",0.757017,0.760030,0.768461,0.761836,0.004843,6
9,1015.853098,2.429072,0.236622,0.005982,0.85,400,0.7,2,0.10,"{'model__subsample': 0.85, 'model__n_estimator...",0.756067,0.760254,0.769080,0.761800,0.005424,7
5,1022.275722,2.908058,0.241833,0.001790,0.85,400,0.9,3,0.10,"{'model__subsample': 0.85, 'model__n_estimator...",0.756348,0.761106,0.767672,0.761709,0.004643,8
14,949.416317,35.077740,0.182978,0.005588,0.85,400,0.7,3,0.10,"{'model__subsample': 0.85, 'model__n_estimator...",0.755013,0.759451,0.768171,0.760879,0.005466,9
2,1654.771430,12.920451,0.260922,0.020389,0.75,700,0.7,2,0.10,"{'model__subsample': 0.75, 'model__n_estimator...",0.754825,0.759909,0.767484,0.760739,0.005201,10


In [29]:
# Best hyperparameters from the search, without the pipeline prefix
best_params = {k.replace('model__', ''): v for k, v in search.best_params_.items()}
model_params = {**best_params, 'random_state': 42}
print(model_params)

# Save them so the run can be reproduced without re-tuning
pd.Series(model_params).to_csv('../outputs/gbm_best_params.csv', header=['value'])

{'subsample': 0.85, 'n_estimators': 700, 'max_features': 0.7, 'max_depth': 4, 'learning_rate': 0.05, 'random_state': 42}


## Test and scoring

In [30]:
df = pd.concat([train_df, test_df])

In [31]:
df.shape

(82739, 200)

In [32]:
y_dtype = [('event', bool), ('duration', float)]

y = np.array([tuple(x) for x in df[['event', 'duration']].values],
             dtype = y_dtype)

In [33]:
cv_scores_df, gbm_feature_importance = cross_validated_survival_predictions(
    df=df,
    y=y, 
    all_var=all_var, 
    preprocessor=preprocessor,
    model_params=model_params, 
    survival_timepoints=(180, 365), 
    tau=365,
    n_splits=3, 
    random_state=42, 
    return_importances=True)

gbm_auc_180d_df, gbm_auc_over5_df = out_of_fold_auc(cv_scores_df, y, model_name='gbm')

cv_scores_df.to_csv('../outputs/gb_survival_predictions.csv')
gbm_feature_importance.to_csv('../outputs/gbm_feature_importance.csv', index=False)
gbm_auc_180d_df.to_csv('../outputs/gbm_auc_180d_df.csv', index=False)
gbm_auc_over5_df.to_csv('../outputs/gbm_auc_over5_df.csv')

Processing fold 1/3
Processing fold 2/3
Processing fold 3/3
Finished fold 1/3
Finished fold 3/3
Finished fold 2/3

Number of patients in df: 82739
risk_scores computed: 82739
psurv_180 computed: 82739
psurv_365 computed: 82739
